# Biomass-floor and ATP-capacity FVA

Edit MODEL_DIR and OUT. Supply external <stage>_c.xml files for eight contextual
models and the separately labelled synthetic L4 union. Dependencies: COBRApy, NumPy,
pandas and Gurobi with a usable license. Relative paths resolve from the kernel
working directory; no parent search or directory change is performed.

Maximize MAR04413 under delivered bounds. At each floor f × optimum, with f = 0, 0.01,
0.10 or 0.25, calculate three scenarios: paper network with a cleared FVA objective;
ATP-assay control with a cleared FVA objective; and ATP-optimal FVA with
fraction_of_optimum=1. The ATP assay is MAR03964 with the exact saved stoichiometry
and bounds 0–1000. It is constructed and added through COBRApy. Biomass and ATP
capacities are maximized separately, not assumed simultaneously achievable. A zero
biomass floor permits biomass; it does not fix it to zero.

All delivered exchange conditions remain unchanged. Ordinary FVA describes feasible
intervals, not sampled distributions or a thermodynamic guarantee. ATP capacity is not
measured maintenance. Fluxes retain model units; declared mmol_per_gDW_per_hr means
mmol gDW⁻¹ h⁻¹, without an independently established biomass-normalization or
division-rate conversion. No analysis was executed for this copy.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cobra
from cobra.flux_analysis import flux_variability_analysis

MODEL_DIR = Path("inputs/models")
OUT = Path("results/objective_sensitivity_results")
OUT.mkdir(parents=True, exist_ok=True)
STAGES = ["egg", "L1", "L2", "L3", "L4f", "L4m", "Am", "Af", "L4mf_union"]
FRACTIONS = [0.0, 0.01, 0.10, 0.25]
FAMILIES = ["paper_network", "ATP_assay_control", "ATP_optimal"]
BIO = "MAR04413"
ATPID = "MAR03964"
ATP_BOUNDS = (0.0, 1000.0)
ATP_STOICHIOMETRY = {
    "MAM01371c": -1.0,
    "MAM02040c": -1.0,
    "MAM01285c": 1.0,
    "MAM02039c": 1.0,
    "MAM02751c": 1.0,
}
SOLVER_TOL = 1e-9
FLUX_TOL = 1e-4
capacity_rows = []
status_rows = []
inventory_rows = []


In [ ]:
for stage in STAGES:
    model = cobra.io.read_sbml_model(str(MODEL_DIR / f"{stage}_c.xml"))
    model.solver = "gurobi"
    model.tolerance = SOLVER_TOL
    model.solver.configuration.timeout = 120
    model.objective = model.reactions.get_by_id(BIO)
    model.objective.direction = "max"
    optimum = float(model.slim_optimize(error_value=np.nan))
    assert model.solver.status == "optimal" and optimum > 0
    for reaction in model.reactions:
        inventory_rows.append(
            dict(
                model=stage,
                reaction_id=reaction.id,
                name=reaction.name,
                subsystem=reaction.subsystem or "Unassigned",
                boundary=reaction.boundary,
            )
        )
    assay = model.copy()
    assert ATPID not in assay.reactions
    atp = cobra.Reaction(ATPID, name="ATP phosphohydrolase")
    atp.bounds = ATP_BOUNDS
    atp.add_metabolites(
        {
            assay.metabolites.get_by_id(rid): coefficient
            for rid, coefficient in ATP_STOICHIOMETRY.items()
        }
    )
    assay.add_reactions([atp])
    for fraction in FRACTIONS:
        for family in FAMILIES:
            scenario = model if family == "paper_network" else assay
            tag = f"{stage}_{family}_{int(fraction * 100):02d}pct"
            with scenario:
                scenario.reactions.get_by_id(BIO).lower_bound = fraction * optimum
                row = dict(
                    model=stage,
                    family=family,
                    biomass_fraction=fraction,
                    reference_biomass_optimum=optimum,
                    biomass_lower_bound=fraction * optimum,
                )
                objectives = [(BIO, "biomass")]
                if family != "paper_network":
                    objectives.append((ATPID, "ATP"))
                for reaction_id, label in objectives:
                    scenario.objective = scenario.reactions.get_by_id(reaction_id)
                    scenario.objective.direction = "max"
                    try:
                        value = float(scenario.slim_optimize(error_value=np.nan))
                        status = str(scenario.solver.status)
                    except Exception as error:
                        value, status = np.nan, "error:" + type(error).__name__
                    row[label + "_maximum"] = value
                    row[label + "_solver_status"] = status
                row["ATP_cap_reached"] = bool(
                    row.get("ATP_solver_status") == "optimal"
                    and row["ATP_maximum"] >= ATP_BOUNDS[1] - FLUX_TOL
                )
                capacity_rows.append(row)
                state, error_text = "not_run_nonoptimal_baseline", ""
                if all(
                    row[label + "_solver_status"] == "optimal"
                    for _, label in objectives
                ):
                    if family == "ATP_optimal":
                        scenario.objective = scenario.reactions.get_by_id(ATPID)
                        objective_fraction = 1.0
                    else:
                        scenario.objective = scenario.problem.Objective(
                            0, direction="max"
                        )
                        objective_fraction = 0.0
                    try:
                        ranges = flux_variability_analysis(
                            scenario,
                            fraction_of_optimum=objective_fraction,
                            loopless=False,
                            processes=1,
                        )
                        if not np.isfinite(
                            ranges[["minimum", "maximum"]].to_numpy()
                        ).all():
                            raise ValueError("Nonfinite FVA endpoints")
                        ranges.rename_axis("reaction_id").reset_index().to_csv(
                            OUT / f"fva_{tag}.tsv", sep="\t", index=False
                        )
                        state = "completed"
                    except Exception as error:
                        state, error_text = "failed", repr(error)
                status_rows.append(
                    dict(
                        model=stage,
                        family=family,
                        biomass_fraction=fraction,
                        status=state,
                        error=error_text,
                    )
                )
    pd.DataFrame(capacity_rows).to_csv(
        OUT / "fba_all_scenarios.tsv", sep="\t", index=False
    )
    pd.DataFrame(status_rows).to_csv(
        OUT / "fva_execution_status.tsv", sep="\t", index=False
    )
    pd.DataFrame(inventory_rows).to_csv(
        OUT / "reaction_inventory.tsv", sep="\t", index=False
    )
